In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv("data_clean.csv")

print("data_clean.csv:")
display(df.head())

data_clean.csv:


,Judul,Jenjang,Tipe (Online/Offline),Penyelenggara,Tema/Kategori,Link Poster,Biaya_Rata_Rata,Tanggal_Mulai,Tanggal_Selesai
0,Pi-Spektra,SD / Sederajat,Online,SMP Daarut Tauhiid Boarding School Putri,"Musik, Agama, English",https://infolomba.id/images/event/poster/pispe...,0,2025-03-03,2025-03-16
1,Medical Veteran Project 2025,Mahasiswa,Online & Offline,BEM FK UPN “Veteran” Jakarta,"Desain, Olahraga, Seni",https://infolomba.id/images/event/poster/medic...,80000,2025-08-17,2025-09-06
2,Mujahidin Excellent Fest 2026,"SD / Sederajat, SMP / Sederajat",Offline,SMP Mujahidin Surabaya,"Olimpiade, Agama",https://infolomba.id/images/event/poster/mujah...,45000,2026-02-05,2026-02-22
3,Lomba Mural Tadjimalela,"SMA / Sederajat, Gapyear, Mahasiswa, Umum",Offline,Mural Tadjimalela,"Desain, Seni, ~Lainnya",https://infolomba.id/images/event/poster/lomba...,100000,2026-01-01,2026-01-25
4,Lomba Video Mengulas Buku!✨📚,"SMP / Sederajat, SMA / Sederajat, Gapyear, Mah...",Online,Yayasan Gelora Insan Mandiri,"~Lainnya, Videografi/Film, Challenge",https://infolomba.id/images/event/poster/lomba...,0,2025-08-11,2025-09-08


### KATEGORI/TEMA MAPPING

ISI-ISI DOMAIN(dari keyword di kolom kategori/tema):

Akademik : 'Esai', 'Olimpiade', 'Karya Tulis Ilmiah', 'Sastra', 'Debat', 'English', 
        'Pidato', 'Akuntansi', 'Media Pembelajaran', 'Beasiswa', 'Artikel', 
        'Hukum', 'Cerdas Cermat', 'Paper', 'Bahasa Asing', 'Try Out', 'Pajak', 
        'Statistika/Data', 'Keuangan', 'Kesehatan'

Seni Kretif : 'Desain', 'Videografi/Film', 'Musik', 'Seni', 'Poster', 'Infografis', 
        'Fotografi', 'Dance/Tari', 'Menggambar/Drawing/Ilustrasi', 'Mewarnai', 
        'Fashion Show', 'Story Telling', 'MC/Protocol', 'Voice Over', 
        'News Anchor/Pembawa Berita', 'Podcast'

Teknologi : 'IT', 'Teknik', 'Robot', 'UI/UX'

Olahraga & E-Sport : 'Olahraga', 'E-sport'

Bisnis : 'Bisnis', 'Seminar', 'Pelatihan', 'Trading', 'Ambassador'

Lainnya : 'Agama', 'Challenge', '~Lainnya', 'Pramuka', 'Rally Games', 
        'Baris Berbaris', 'Giveaway', 'PMR', 'Permainan', 'Lainnya'

In [6]:
# CELL 4: Grouping Domain Kategori (Advanced Level)

# 1. Definisi Mapping Kategori Induk (Super Domain)
# Semua 56 kategori di datamu sudah dimasukkan ke kelompok yang relevan
domain_mapping = {
    'Domain_Akademik': [
        'Esai', 'Olimpiade', 'Karya Tulis Ilmiah', 'Sastra', 'Debat', 'English', 
        'Pidato', 'Akuntansi', 'Media Pembelajaran', 'Beasiswa', 'Artikel', 
        'Hukum', 'Cerdas Cermat', 'Paper', 'Bahasa Asing', 'Try Out', 'Pajak', 
        'Statistika/Data', 'Keuangan', 'Kesehatan'
    ],
    'Domain_Seni_Kreatif': [
        'Desain', 'Videografi/Film', 'Musik', 'Seni', 'Poster', 'Infografis', 
        'Fotografi', 'Dance/Tari', 'Menggambar/Drawing/Ilustrasi', 'Mewarnai', 
        'Fashion Show', 'Story Telling', 'MC/Protocol', 'Voice Over', 
        'News Anchor/Pembawa Berita', 'Podcast'
    ],
    'Domain_Teknologi': [
        'IT', 'Teknik', 'Robot', 'UI/UX'
    ],
    'Domain_Olahraga_E-Sport': [
        'Olahraga', 'E-sport'
    ],
    'Domain_Bisnis_Karir': [
        'Bisnis', 'Seminar', 'Pelatihan', 'Trading', 'Ambassador'
    ],
    'Domain_Umum_Lainnya': [
        'Agama', 'Challenge', '~Lainnya', 'Pramuka', 'Rally Games', 
        'Baris Berbaris', 'Giveaway', 'PMR', 'Permainan', 'Lainnya'
    ]
}

# Bikin dictionary lookup agar mesin cepat mencocokkan katanya
# (Contoh: 'UI/UX' -> 'Domain_Teknologi')
lookup_dict = {}
for domain, kats in domain_mapping.items():
    for k in kats:
        lookup_dict[k] = domain

# 2. Fungsi untuk mengubah deretan kategori mentah menjadi Super Domain
def map_to_domain(teks_kategori):
    teks_kategori = str(teks_kategori).replace(', ', ',')
    kategori_list = teks_kategori.split(',')
    
    domain_terpilih = set() # Pakai set agar tidak ada duplikat domain di 1 lomba
    for kat in kategori_list:
        kat = kat.strip()
        if kat in lookup_dict:
            domain_terpilih.add(lookup_dict[kat])
        elif kat: # Kalau misal ada kategori aneh yang terlewat
            domain_terpilih.add('Domain_Umum_Lainnya') 
            
    # Gabungkan kembali pakai koma (contoh: "Domain_Akademik,Domain_Teknologi")
    return ','.join(list(domain_terpilih))

# 3. Aplikasikan ke kolom Tema/Kategori
df['Super_Kategori'] = df['Tema/Kategori'].apply(map_to_domain)

# 4. Lakukan Multi-Label Binarization pada Super Kategori
domain_dummies = df    ['Super_Kategori'].str.get_dummies(sep=',')

# 5. Gabungkan kolom-kolom baru tersebut ke DataFrame utama
df = pd.concat([df, domain_dummies], axis=1)

# 6. Hapus kolom Tema/Kategori dan Super_Kategori karena sudah jadi angka 0 & 1
df = df.drop(columns=['Tema/Kategori', 'Super_Kategori'])

# Cek hasilnya
print("\n✅ Pemetaan Domain Berhasil!")
print("Bentuk data sekarang:", df.shape)
display(df.head())


✅ Pemetaan Domain Berhasil!
Bentuk data sekarang: (1492, 14)


,Judul,Jenjang,Tipe (Online/Offline),Penyelenggara,Link Poster,Biaya_Rata_Rata,Tanggal_Mulai,Tanggal_Selesai,Domain_Akademik,Domain_Bisnis_Karir,Domain_Olahraga_E-Sport,Domain_Seni_Kreatif,Domain_Teknologi,Domain_Umum_Lainnya
0,Pi-Spektra,SD / Sederajat,Online,SMP Daarut Tauhiid Boarding School Putri,https://infolomba.id/images/event/poster/pispe...,0,2025-03-03,2025-03-16,1,0,0,1,0,1
1,Medical Veteran Project 2025,Mahasiswa,Online & Offline,BEM FK UPN “Veteran” Jakarta,https://infolomba.id/images/event/poster/medic...,80000,2025-08-17,2025-09-06,0,0,1,1,0,0
2,Mujahidin Excellent Fest 2026,"SD / Sederajat, SMP / Sederajat",Offline,SMP Mujahidin Surabaya,https://infolomba.id/images/event/poster/mujah...,45000,2026-02-05,2026-02-22,1,0,0,0,0,1
3,Lomba Mural Tadjimalela,"SMA / Sederajat, Gapyear, Mahasiswa, Umum",Offline,Mural Tadjimalela,https://infolomba.id/images/event/poster/lomba...,100000,2026-01-01,2026-01-25,0,0,0,1,0,1
4,Lomba Video Mengulas Buku!✨📚,"SMP / Sederajat, SMA / Sederajat, Gapyear, Mah...",Online,Yayasan Gelora Insan Mandiri,https://infolomba.id/images/event/poster/lomba...,0,2025-08-11,2025-09-08,0,0,0,1,0,1


### Memfokuskan kolom jenjang

Jika jenjangnya lebih dari 3 (berarti 4, 5, dst), ubah jadi 'Umum' dan Jika jenjangnya 1, 2, atau 3, ambil elemen yang paling pertama [0]

In [7]:
# Fungsi untuk menyederhanakan kolom Jenjang
def rapihin_jenjang(teks):
    teks = str(teks)
    # Pecah teks berdasarkan koma dan hapus spasi berlebih
    parts = [p.strip() for p in teks.split(',')]
    
    # Jika jenjangnya lebih dari 3 (berarti 4, 5, dst), ubah jadi 'Umum'
    if len(parts) > 3:
        return 'Umum'
    # Jika jenjangnya 1, 2, atau 3, ambil elemen yang paling pertama [0]
    else:
        return parts[0]

# 1. Aplikasikan fungsinya untuk menimpa kolom Jenjang yang lama
df['Jenjang'] = df['Jenjang'].apply(rapihin_jenjang)

# 2. Karena isinya sudah sangat rapi dan seragam, kita ubah lagi jadi tipe 'category' 
# (biar hemat memori dan cepat diolah)
df['Jenjang'] = df['Jenjang'].astype('category')

# 3. Cek hasil perubahannya
print("Preview Jenjang yang sudah dirapikan:")
display(df[['Judul', 'Jenjang']].head(10))

# Jangan lupa save ke CSV akhir kalau ini udah tahap paling final ya!
# df.to_csv('data_clean_final.csv', index=False)

Preview Jenjang yang sudah dirapikan:


,Judul,Jenjang
0,Pi-Spektra,SD / Sederajat
1,Medical Veteran Project 2025,Mahasiswa
2,Mujahidin Excellent Fest 2026,SD / Sederajat
3,Lomba Mural Tadjimalela,Umum
4,Lomba Video Mengulas Buku!✨📚,Umum
5,FusionTech Challenge 2025,SMA / Sederajat
6,Olimpiade Manajemen UNAIR 2025,SMA / Sederajat
7,Lomba MC Nasional,Umum
8,ALKAFEST 2025,Umum
9,Pharmasteria 2025,SMA / Sederajat


#### Encoding yang jenjang

'SD / Sederajat': 1,
    'SMP / Sederajat': 2,
    'SMA / Sederajat': 3,
    'Mahasiswa': 4,
    'Umum': 5

In [8]:
# Membuat urutan hierarki secara manual (Ordinal Encoding)
mapping_jenjang = {
    'SD / Sederajat': 1,
    'SMP / Sederajat': 2,
    'SMA / Sederajat': 3,
    'Mahasiswa': 4,
    'Umum': 5  # Umum ditaruh tertinggi karena cakupannya paling luas
}

# Terapkan Label Encoding ke kolom Jenjang
df['Jenjang_Encoded'] = df['Jenjang'].map(mapping_jenjang)

# Lihat hasilnya
display(df[['Judul', 'Jenjang', 'Jenjang_Encoded']].head(10))

,Judul,Jenjang,Jenjang_Encoded
0,Pi-Spektra,SD / Sederajat,1.0
1,Medical Veteran Project 2025,Mahasiswa,4.0
2,Mujahidin Excellent Fest 2026,SD / Sederajat,1.0
3,Lomba Mural Tadjimalela,Umum,5.0
4,Lomba Video Mengulas Buku!✨📚,Umum,5.0
5,FusionTech Challenge 2025,SMA / Sederajat,3.0
6,Olimpiade Manajemen UNAIR 2025,SMA / Sederajat,3.0
7,Lomba MC Nasional,Umum,5.0
8,ALKAFEST 2025,Umum,5.0
9,Pharmasteria 2025,SMA / Sederajat,3.0


In [9]:
# Ubah kembali kedua kolom tanggal menjadi tipe datetime
df['Tanggal_Mulai'] = pd.to_datetime(df['Tanggal_Mulai'])
df['Tanggal_Selesai'] = pd.to_datetime(df['Tanggal_Selesai'])

# Cek tipe datanya sekarang
print(df[['Tanggal_Mulai', 'Tanggal_Selesai']].dtypes)

Tanggal_Mulai      datetime64[ns]
Tanggal_Selesai    datetime64[ns]
dtype: object


### Encoding Tipe(online/offline)

Kalau lombanya "Online" ➡️ Is_Online = 1, Is_Offline = 0

Kalau lombanya "Offline" ➡️ Is_Online = 0, Is_Offline = 1

Kalau lombanya "Online & Offline" ➡️ Is_Online = 1, Is_Offline = 1 (Karena mendukung keduanya!)

In [10]:
# Membuat kolom Is_Online (Kasih nilai 1 jika ada kata 'Online')
df['Is_Online'] = df['Tipe (Online/Offline)'].apply(lambda x: 1 if 'Online' in str(x) else 0)

# Membuat kolom Is_Offline (Kasih nilai 1 jika ada kata 'Offline')
df['Is_Offline'] = df['Tipe (Online/Offline)'].apply(lambda x: 1 if 'Offline' in str(x) else 0)

# Hapus kolom aslinya karena sudah diwakili oleh 2 kolom angka di atas
df = df.drop(columns=['Tipe (Online/Offline)'])

# Cek hasil perubahannya
print("Preview perubahan Tipe Lomba:")
display(df[['Judul', 'Is_Online', 'Is_Offline']].head(10))

Preview perubahan Tipe Lomba:


,Judul,Is_Online,Is_Offline
0,Pi-Spektra,1,0
1,Medical Veteran Project 2025,1,1
2,Mujahidin Excellent Fest 2026,0,1
3,Lomba Mural Tadjimalela,0,1
4,Lomba Video Mengulas Buku!✨📚,1,0
5,FusionTech Challenge 2025,1,1
6,Olimpiade Manajemen UNAIR 2025,1,1
7,Lomba MC Nasional,1,0
8,ALKAFEST 2025,0,1
9,Pharmasteria 2025,1,1


In [11]:
df.to_csv('hasil_feature.csv', index=False)